# Sharpe vs Volatility — Binned Density per Block

Computes monthly Sharpe ratio and monthly volatility for every wallet in selected blocks,
bins into a 2D grid, and exports a lightweight CSV for R plotting.

**Scaling (monthly, 30-day):**
- `vol_monthly = sqrt(variance_daily) * sqrt(30)`
- `ret_monthly = expected_return_daily * 30`
- `sharpe_monthly = ret_monthly / vol_monthly`

In [33]:
import duckdb
import numpy as np
import time

In [34]:
# --- CONFIG ---
import sys
sys.path.insert(0, "../../shared")
from locations import Location

ROOT = str(Location.MPT_DATA)
OUT_CSV = "sharpe_vs_vol_binned.csv"

# Pick your blocks here (block_number values)
# Examples: early 2020, bull peak 2021, crash 2022, recovery 2023, mid 2024, late 2025
SELECTED_BLOCKS = [
    9193266,   # Jan 2020
    13330090,  # Jan 2022
    15053226,  # Jan 2023
    18037988,  # Jan 2024
    20207949,  # Jan 2025
    23914921,  # Dec 2025
]

# Binning parameters
VOL_MIN, VOL_MAX, VOL_STEP = 0.0, 2.0, 0.02       # monthly vol from 0 to 200%
SHARPE_MIN, SHARPE_MAX, SHARPE_STEP = -3.0, 3.0, 0.1  # monthly Sharpe

print(f"Blocks selected: {len(SELECTED_BLOCKS)}")
print(f"Vol bins: {int((VOL_MAX - VOL_MIN) / VOL_STEP)}, Sharpe bins: {int((SHARPE_MAX - SHARPE_MIN) / SHARPE_STEP)}")

Blocks selected: 6
Vol bins: 100, Sharpe bins: 60


In [35]:
con = duckdb.connect(database=':memory:')

con.execute(f"""
    CREATE VIEW all_data AS
    SELECT * FROM read_parquet('{ROOT}/*/*.parquet', hive_partitioning=true)
""")

# Verify selected blocks exist and show their dates + row counts
block_list = ', '.join(str(b) for b in SELECTED_BLOCKS)
info = con.execute(f"""
    SELECT block_number, date, COUNT(*) AS n_wallets
    FROM all_data
    WHERE block_number IN ({block_list})
    GROUP BY block_number, date
    ORDER BY block_number
""").fetchdf()

display(info)
print(f"Total wallets to process: {info['n_wallets'].sum():,}")

,block_number,date,n_wallets
0,9193266,2020-01-01,879029
1,13330090,2021-10-01,2386807
2,15053226,2022-07-01,3104578
3,18037988,2023-09-01,3678201
4,20207949,2024-07-01,4583434
5,23914921,2025-12-01,7112115


Total wallets to process: 21,744,164


In [36]:
# --- COMPUTE SHARPE & VOL, THEN BIN ---
t0 = time.time()
print("Computing monthly Sharpe and volatility, binning into 2D grid ...")

df_binned = con.execute(f"""
    WITH metrics AS (
        SELECT
            block_number,
            date,
            initial_variance_daily,
            initial_expected_return_daily,
            -- Monthly scaling (30 days)
            SQRT(initial_variance_daily) * SQRT(30)   AS vol_monthly,
            initial_expected_return_daily * 30         AS ret_monthly
        FROM all_data
        WHERE block_number IN ({block_list})
          AND initial_variance_daily > 0
          AND initial_variance_daily IS NOT NULL
          AND initial_expected_return_daily IS NOT NULL
    ),
    with_sharpe AS (
        SELECT
            block_number,
            date,
            vol_monthly,
            ret_monthly,
            CASE WHEN vol_monthly > 0 THEN ret_monthly / vol_monthly ELSE 0 END AS sharpe_monthly
        FROM metrics
    ),
    binned AS (
        SELECT
            block_number,
            date,
            -- Clamp and bin volatility
            FLOOR(LEAST(GREATEST(vol_monthly, {VOL_MIN}), {VOL_MAX} - 0.0001) / {VOL_STEP}) * {VOL_STEP} AS vol_bin,
            -- Clamp and bin Sharpe
            FLOOR(LEAST(GREATEST(sharpe_monthly, {SHARPE_MIN}), {SHARPE_MAX} - 0.0001) / {SHARPE_STEP}) * {SHARPE_STEP} AS sharpe_bin
        FROM with_sharpe
    )
    SELECT
        block_number,
        date,
        ROUND(vol_bin, 4)    AS vol_bin,
        ROUND(sharpe_bin, 4) AS sharpe_bin,
        COUNT(*)             AS n_wallets
    FROM binned
    GROUP BY block_number, date, vol_bin, sharpe_bin
    ORDER BY block_number, vol_bin, sharpe_bin
""").fetchdf()

print(f"Done in {time.time() - t0:.1f}s")
print(f"Result: {len(df_binned):,} bins across {df_binned['block_number'].nunique()} blocks")
print(f"Total wallets represented: {df_binned['n_wallets'].sum():,}")

Computing monthly Sharpe and volatility, binning into 2D grid ...
Done in 1.4s
Result: 16,784 bins across 6 blocks
Total wallets represented: 21,744,164


In [37]:
# --- PREVIEW ---
print("\nBin counts per block:")
summary = df_binned.groupby(['block_number', 'date']).agg(
    n_bins=('n_wallets', 'count'),
    total_wallets=('n_wallets', 'sum')
).reset_index()
display(summary)

print("\nSample rows (top-populated bins):")
display(df_binned.nlargest(15, 'n_wallets'))


Bin counts per block:


,block_number,date,n_bins,total_wallets
0,9193266,2020-01-01,1696,879029
1,13330090,2021-10-01,2276,2386807
2,15053226,2022-07-01,2683,3104578
3,18037988,2023-09-01,3210,3678201
4,20207949,2024-07-01,3385,4583434
5,23914921,2025-12-01,3534,7112115



Sample rows (top-populated bins):


,block_number,date,vol_bin,sharpe_bin,n_wallets
13273,23914921,2025-12-01,0.00,0.0,548620
13274,23914921,2025-12-01,0.00,0.1,244911
4730,15053226,2022-07-01,0.40,-1.0,215618
2241,13330090,2021-10-01,0.30,-0.2,202998
10623,20207949,2024-07-01,0.32,0.2,201636
7554,18037988,2023-09-01,0.36,0.5,199304
13919,23914921,2025-12-01,0.28,-0.7,180286
933,9193266,2020-01-01,0.60,-0.4,171390
14017,23914921,2025-12-01,0.32,-0.7,134797
6679,18037988,2023-09-01,0.00,-0.1,129214


In [38]:
# --- QUICK STATS per block ---
for block in SELECTED_BLOCKS:
    subset = df_binned[df_binned['block_number'] == block]
    if subset.empty:
        print(f"\nBlock {block}: no data")
        continue
    total = subset['n_wallets'].sum()
    # Weighted stats
    avg_vol = np.average(subset['vol_bin'] + VOL_STEP/2, weights=subset['n_wallets'])
    avg_sharpe = np.average(subset['sharpe_bin'] + SHARPE_STEP/2, weights=subset['n_wallets'])
    date = subset['date'].iloc[0]
    print(f"\nBlock {block} ({date}): {total:,} wallets")
    print(f"  Weighted avg vol (monthly):    {avg_vol:.4f} ({avg_vol*100:.1f}%)")
    print(f"  Weighted avg Sharpe (monthly): {avg_sharpe:.3f}")


Block 9193266 (2020-01-01 00:00:00): 879,029 wallets
  Weighted avg vol (monthly):    0.4350 (43.5%)
  Weighted avg Sharpe (monthly): -0.588

Block 13330090 (2021-10-01 00:00:00): 2,386,807 wallets
  Weighted avg vol (monthly):    0.3516 (35.2%)
  Weighted avg Sharpe (monthly): 0.089

Block 15053226 (2022-07-01 00:00:00): 3,104,578 wallets
  Weighted avg vol (monthly):    0.3893 (38.9%)
  Weighted avg Sharpe (monthly): -0.800

Block 18037988 (2023-09-01 00:00:00): 3,678,201 wallets
  Weighted avg vol (monthly):    0.2556 (25.6%)
  Weighted avg Sharpe (monthly): -0.278

Block 20207949 (2024-07-01 00:00:00): 4,583,434 wallets
  Weighted avg vol (monthly):    0.2688 (26.9%)
  Weighted avg Sharpe (monthly): -0.271

Block 23914921 (2025-12-01 00:00:00): 7,112,115 wallets
  Weighted avg vol (monthly):    0.2969 (29.7%)
  Weighted avg Sharpe (monthly): -0.593


In [39]:
# --- SAVE ---
df_binned.to_csv(OUT_CSV, index=False)
print(f"Saved: {OUT_CSV}  ({len(df_binned):,} rows)")

Saved: sharpe_vs_vol_binned.csv  (16,784 rows)


In [40]:
con.close()